# 电路中的电压和电流

scikit-rf 的 `Circuit` 对象允许我们在给定功率（和相位）激励下推导电路所有交点处的电压和电流。这里，我们分析几个示例，以阐明用于电流方向的约定。

In [ ]:
%matplotlib inline

In [ ]:
# standard imports
import numpy as np

import skrf as rf
from skrf.circuit import Circuit

In [ ]:
rf.stylely()

## 简单的传输线

首先，假设有一条由激励源激发的简单传输线。假设源与传输线匹配（$Z_s=Z_0$），传输线连接到一个匹配负载（$Z_0=Z_L$），全部为 50 欧姆。

<img src="circuit_vi_simple_line.svg">

对于给定的输入功率（和相位），这条传输线的输入和输出处的射频电流和电压是多少？

In [ ]:
P_f = 1  # forward power in Watt
Z = 50  # source internal impedance, line characteristic impedance and load impedance
L = 10  # line length in [m]

freq = rf.Frequency(2, 2, 1, unit='GHz')
line_media = rf.media.DefinedGammaZ0(freq, z0=Z)  # lossless line medium
line = line_media.line(d=L, unit='m', name='line')  # transmission line Network

假设源产生相位为 $\phi$ 的输入功率 $P_f$，对于这样的传输线，传输线入口处的电压和电流为：

$$
V_{in} = \sqrt{2 Z_0 P_f} e^{j \phi}
$$

$$
I_{in} = \sqrt{2 \frac{P_f}{Z_0}} e^{j \phi}
$$

In [ ]:
V_in = np.sqrt(2*Z*P_f)
I_in = np.sqrt(2*P_f/Z)
print(f'Input voltage and current: {V_in} V and {I_in} A')

电压和电流沿传输线演变。可以使用 scikit-rf 提供的传输线工具计算传输线输出处的电压和电流：

In [ ]:
theta = rf.tlineFunctions.theta(line_media.gamma, freq.f, L)  # electrical length
V_out, I_out = rf.tlineFunctions.voltage_current_propagation(V_in, I_in, Z, theta)
print(f'Output voltage and current: {V_out} V and {I_out} A')

让我们使用 `Circuit` 执行相同的计算。首先，需要定义电路，即创建输入/输出端口并将这些端口连接到我们已经创建的传输线网络。然后，我们可以构建电路：

In [ ]:
port1 = Circuit.Port(frequency=freq, name='port1', z0=50)
port2 = Circuit.Port(frequency=freq, name='port2', z0=50)
cnx = [
    [(port1, 0), (line, 0)],
    [(port2, 0), (line, 1)]
]
crt = Circuit(cnx)

检查电路图形是否符合预期是一个好习惯。在这种情况下，图形非常简单：2 个端口连接到一个 2 端口网络。

In [ ]:
crt.plot_graph(network_labels=True, edge_labels=True, inter_labels=True)

`Circuit` 提供了两种方法来确定电路输入/输出端口（也称为"外部端口"）处的电压和电流。这些方法以每个端口的功率和相位输入为参数：

In [ ]:
power = [1, 0]  # 1 Watt at port1 and 0 at port2
phase = [0, 0]  # 0 radians

In [ ]:
V_at_ports = crt.voltages_external(power, phase)
print(V_at_ports)

In [ ]:
I_at_ports = crt.currents_external(power, phase)
print(I_at_ports)

结果与之前的类似，除了端口2的电流符号相反。

这是正常的，因为 `currents_external()` 方法将电流的正方向定义为"进入"电路的方向。关于此约定的更多细节见本示例的底部。

## 特征阻抗不同的同轴 T 型接头

作为一个更高级的示例，我们在全波软件（这里是 ANSYS HFSS，但其他软件也可以）中构建了以下结构：同轴 T 型接头，使用不同的同轴横截面（因此特征阻抗不同）。

<img src="circuit_vi_HFSS_Coaxial_T.png" with="800">

所有三个端口都以不同的功率和相位激励，以增加复杂性，如下表所示：

| 端口 | 功率 [W] | 相位 [deg] |
| --- | --- | --- |
| #1 | 1 | -10 |
| #2 | 2 | -20 |
| #3 | 3 | +60 |

### 全波参考解

在全波软件中直接计算电压和电流。电压通过将 E 场沿从端口横截面内导体到外导体的直线积分得到。电流通过将 H 场沿包围端口横截面内导体的圆周积分得到。注意这些圆的方向是为了将正电流方向定义为向端口内部的方向（右手定则）。

全波解在此作为参考，在 3 个频率处给出：

In [ ]:
import pandas as pd  # convenient to read .csv files

pd.read_csv('circuit_vi_HFSS_Voltages.csv')

In [ ]:
pd.read_csv('circuit_vi_HFSS_Currents.csv')

### 在电路仿真器中

电压和电流也可以使用电路仿真器推导，例如：

<img src="circuit_vi_Designer_circuit.png" width="800">

（同样，电流探头的方向使得电流在流入网络时为正）。这给出了基本相同的结果：

In [ ]:
pd.read_csv('circuit_vi_Designer_Voltages.csv')

In [ ]:
pd.read_csv('circuit_vi_Designer_Currents.csv')

### 使用 `Circuit`

现在让我们使用 scikit-rf 的 `Circuit` 构建相同的电路：

In [ ]:
# Importing the 3-port .s3p file exported from full-wave simulation
coaxial_T = rf.Network('circuit_vi_Coaxial_T.s3p')
# pay attention to the port's characteristic impedance
# it should match the Network characteristic impedances otherwise this will generate mismatches
port1 = Circuit.Port(coaxial_T.frequency, 'port1', coaxial_T.z0[:,0])
port2 = Circuit.Port(coaxial_T.frequency, 'port2', coaxial_T.z0[:,1])
port3 = Circuit.Port(coaxial_T.frequency, 'port3', coaxial_T.z0[:,2])
# connection list
cnx = [
    [(port1, 0), (coaxial_T, 0)],
    [(port2, 0), (coaxial_T, 1)],
    [(port3, 0), (coaxial_T, 2)]
]
# building the circuit
crt = Circuit(cnx)

In [ ]:
# let's check if our connection list is correctly defined
crt.plot_graph(network_labels=True, edge_labels=True, inter_labels=True)

对于给定激励，端口处的电压和电流为：

In [ ]:
power = [1, 2, 3]  # input power in watts at ports 1, 2 and 3
phase = np.deg2rad([-10, -20, +60])  # input phase in rad at ports 1, 2 and 3

voltages = crt.voltages_external(power, phase)
currents = crt.currents_external(power, phase)

In [ ]:
# just for a better rendering in the notebook
pd.concat([
    pd.DataFrame(np.abs(voltages), columns=['mag V1', 'mag V2', 'mag V3'], index=crt.frequency.f/1e6),
    pd.DataFrame(np.angle(voltages, deg=True), columns=['Phase V1', 'Phase V2', 'Phase V3'], index=crt.frequency.f/1e6)
], axis=1)

In [ ]:
# just for a better rendering in the notebook
pd.concat([
    pd.DataFrame(np.abs(currents), columns=['mag I1', 'mag I2', 'mag I3'], index=crt.frequency.f/1e6),
    pd.DataFrame(np.angle(currents, deg=True), columns=['Phase I1', 'Phase I2', 'Phase I3'], index=crt.frequency.f/1e6)
], axis=1)

这些结果与全波计算的结果很好地匹配，万岁。

## 外部端口 vs 内部端口？
你可能在 `Circuit` 文档中注意到，我们经常讨论"内部"或"内"端口，以及"外部"或"外"端口。外部端口对应于构建 `Circuit` 时定义的 `Circuit.Port()` 网络。内部端口是 `Circuit` 内部所有的其他连接

`Circuit` 算法允许访问电路内部连接处的电压和电流。在之前的示例中，由于我们直接将 N 端口连接到外部端口，所以内部端口不多。然而，在更复杂的用法中，我们可能会有很多内部端口（查看其他 `Circuit` 示例）。

在 `Circuit` 中，电压和电流是峰值。虽然电压以非歧义的方式定义，但正电流可以用一种或另一种方式定义，导致 180 度的模糊性。为了解决这个问题，我们选择了以下定义：内部电流定义为在流入网络时测量为正的值。

因此，你会发现"外部"电流与相应端口的"内部"电流符号相反，因为内部电流实际上流向端口网络。

In [ ]:
# internals currents (currents at all connexions)
# in this example, there are 6 internal connexions (3 pairs of connexions)
crt.currents(power, phase)

In [ ]:
# This gives the indices of the "external" ports
crt.port_indexes

In [ ]:
# So we can keep only "external" ports
crt.currents(power, phase)[:, crt.port_indexes]

In [ ]:
# note the sign difference due to the convention chosen for internal ports
crt.currents_external(power, phase)

下图使用之前的示例说明这些差异：

<img src="circuit_vi_convention_currents.png" width="800">

## 复杂电路中的电压和电流
### 功率波和行波

在 `scikit-rf` 中，`Voltages` 和 `Currents` 的计算依赖于 `Circuit` 的散射参数（`S-parameters`）。`S-parameters` 的默认定义使用[_功率波_](https://scikit-rf.readthedocs.io/en/latest/examples/networktheory/Working%20with%20Complex%20Characteristic%20Impedances.html)，表示为：
$$ s_{ij} = \frac{b_i}{a_j} $$
其中 $b_i$ 和 $a_j$ 分别表示给定 `Network` 中端口 `i` 和 `j` 的输出和输入功率波。

作为示例，`Circuit._b()` 和 `Circuit._a()` 方法提供支持以计算这些输出和输入功率波，尽管这对于典型用例来说通常是不必要的。

In [ ]:
# To obtain the input/output power waves under given phase and amplitude, you can use:
a = crt._a(crt._a_external(power, phase))
b = crt._b(a)

从 `S-parameters` 的定义，我们可以确定给定 `Network` 中端口 `i` 和 `j` 的前向（输入）行波 $V^+_i$ 和后向（输出）行波 $V^-_j$：

$$V_i^+ = a_i \sqrt{\Re({Z_0}_i)} $$
$$V_j^- = b_j \sqrt{\Re({Z_0}_j)} $$

这里，${Z_0}_i$ 和 ${Z_0}_j$ 分别是端口 `i` 和 `j` 的特征阻抗。

### 电压和行波

在传输线理论中，总电压和总电流是入射和反射电压波幅度的函数。类似地，`Circuit` 中每个节点的 `Voltage` 和 `Current` 可以使用形成该节点的元件端口上的前向（输入）和后向（输出）行波来计算。

对于简单的串联连接，例如当 `Network` $A$ 的端口 `m` 连接到 `Network` $B$ 的端口 `n` 时，节点上的总 `Voltage` 可以使用与传输线相同的原理计算：

$$V=V_m^- + V_n^-$$

这里，$V_m^-$ 和 $V_n^-$ 分别是端口 `m` 和 `n` 上的后向（输出）行波。

### 阻抗失配下的电压

当 `Network` $A$ 的端口 `m`（${Z_0}_{Am}$）和 `Network` $B$ 的端口 `n`（${Z_0}_{Bn}$）的端口特征阻抗不同时，发生阻抗失配。这需要使用传输系数 $\Tau_{nm}$ 来修正失配。

当电压波从 `Network` $A$ 的端口 `m` 传输到 `Network` $B$ 的端口 `n` 时，`Network` $A$ 可以被视为源，`Network` $B$ 视为负载。然后可以使用从 `m` 到 `n` 的传输系数 $\Tau_{nm}$ 来调整失配：

$$ \Tau_{nm} = \frac{2 Z_{Load}}{Z_{Load}+Z_{Source}} = \frac{2 {Z_0}_{Am}}{{Z_0}_{Am}+{Z_0}_{Bn}}$$

### 具有并联连接的复杂电路

在更复杂的 `Circuits` 中，例如具有并联连接的电路，多个端口可能汇聚于一个节点，每个端口可能具有不同的特征阻抗。尽管增加了复杂性，但使用传输系数 $\Tau$ 的方法仍然有效。

通过选择一个 `Network` 的特征阻抗 $Z_l$ 作为源阻抗，并将其他 $k$ 个并联 `Network` 的组合特征阻抗视为负载阻抗，可以确定端口 `l` 的实际传输系数：

$$ \frac{1}{Z_{Source}}=\sum_{i=0}^{k}{\frac{1}{Z_{i}}}-\frac{1}{Z_l}$$

这种方法允许计算节点处的有效阻抗，然后可以用于调整阻抗失配，并确保在复杂的并联 `Network` 配置中准确计算 `Voltage` 和 `Current`。

### 示例

[Wilkinson Power Divider](https://scikit-rf.readthedocs.io/en/latest/examples/circuit/Wilkinson%20Power%20Splitter.html) 提供了复杂电路中电压和电流计算的示例。它演示了使用分路器进行纯串联连接的 `Circuit` 和涉及元件并联化的更复杂 `Circuit` 的计算，在两种情况下都产生一致的结果。